In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ==================== 设备配置 ====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
print(f"使用设备: {device}")

# ==================== 几何参数 ====================
d0 = 0.05  # 内管直径（m）
d1 = 0.13  # 外壳直径（m）
r0 = d0 / 2  # 内管半径
r1 = d1 / 2  # 外壳半径
L_char = d1 - d0  # 特征长度：环形域宽度 (m)

# ==================== 材料参数 ====================
# PCM（石蜡）- 来自论文Table 1
rho_s = 880.0      # 固相密度 (kg/m³)
rho_l = 760.0      # 液相密度 (kg/m³)
cp_s = 2180.0      # 固相定压比热容 (J/(kg·K))
cp_l = 2390.0      # 液相定压比热容 (J/(kg·K))
lambda_s = 0.4     # 固相导热系数 (W/(m·K))
lambda_l = 0.15    # 液相导热系数 (W/(m·K))
mu_l = 0.001       # 液相粘度 (kg/(m·s))
L = 255000.0       # 相变潜热 (J/kg)
Tpc = 316.15       # 相变温度 (K)
DeltaT = 6.0       # 相变温度区间 (K)
alpha = 1.0e-4     # 体膨胀系数 (1/K)
g = 9.81           # 重力加速度 (m/s²)

# 高导热材料（铜）
rho_Cu = 8960.0    # 密度 (kg/m³)
lambda_Cu = 400.0  # 导热系数 (W/(m·K))
cp_Cu = 385.0      # 定压比热容 (J/(kg·K))

# ==================== 修正的特征尺度计算 ====================
def compute_characteristic_scales():
    """基于物理分析计算合理的特征尺度 - 修正版"""
    # 特征温差：根据论文，加热温度360K，初始290K，相变316K
    # 使用较小的特征温差确保数值稳定性
    DeltaT_scale = 50.0  # 修正的特征温差
    
    # 特征速度（基于自然对流 - 修正公式）
    # U_char = sqrt(g * beta * DeltaT * L_char) - 但需要确保数值稳定性
    U_char = np.sqrt(g * alpha * DeltaT_scale * L_char)  # 自然对流速度尺度
    
    # 如果速度过小或过大，进行限制
    U_char = np.clip(U_char, 0.001, 0.1)
    
    # 特征时间（基于热扩散时间）
    # 使用热扩散时间而非对流时间，更稳定
    alpha_thermal = lambda_s / (rho_s * cp_s)  # 热扩散系数
    t_char = L_char**2 / alpha_thermal  # 热扩散时间尺度
    
    # 特征压力
    p_char = rho_s * U_char**2
    
    print("="*60)
    print("修正的特征尺度:")
    print(f"特征长度 L_char = {L_char:.4f} m")
    print(f"特征速度 U_char = {U_char:.6f} m/s")
    print(f"特征时间 t_char = {t_char:.2f} s")
    print(f"特征温差 T_char = {DeltaT_scale:.1f} K")
    print(f"特征压力 p_char = {p_char:.6f} Pa")
    print("="*60)
    
    return {
        'L_char': L_char,
        'U_char': U_char,
        't_char': t_char,
        'T_char': DeltaT_scale,
        'p_char': p_char
    }

char_scales = compute_characteristic_scales()
L_char = char_scales['L_char']
U_char = char_scales['U_char']
t_char = char_scales['t_char']
T_char = char_scales['T_char']
p_char = char_scales['p_char']

# ==================== 拓扑优化参数 ====================
phi_total = 0.3  # 高导热材料体积比约束
case = 1         # 优化目标选择：1=平均温度，2=温度均方差，3=多目标

# ==================== 数值稳定性参数 ====================
eps = 1e-8      # 防止除零的小量

# ==================== 改进的物理场网络 ====================
class StablePhysicsInformedNN(nn.Module):
    """稳定的物理信息神经网络 - 防止数值爆炸"""
    
    def __init__(self, input_dim=3, hidden_dim=128, num_layers=6):
        super(StablePhysicsInformedNN, self).__init__()
        
        # 构建网络 - 使用更稳定的架构
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.Tanh())
        
        for _ in range(num_layers - 2):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        
        layers.append(nn.Linear(hidden_dim, 4))
        
        self.net = nn.Sequential(*layers)
        
        # 特征尺度
        self.register_buffer('L_char', torch.tensor(L_char))
        self.register_buffer('U_char', torch.tensor(U_char))
        self.register_buffer('t_char', torch.tensor(t_char))
        self.register_buffer('T_char', torch.tensor(T_char))
        self.register_buffer('p_char', torch.tensor(p_char))
        
        # 参考温度
        self.register_buffer('T0', torch.tensor(290.0))  # 初始温度
        self.register_buffer('Tw_heat', torch.tensor(360.0))  # 储热时内壁温度
        self.register_buffer('Tw_cool', torch.tensor(290.0))  # 放热时内壁温度
        
        # 相变温度
        self.register_buffer('Tpc', torch.tensor(Tpc))
        
        # 初始化权重 - 使用更稳定的初始化
        self._initialize_weights()
    
    def _initialize_weights(self):
        """初始化网络权重 - 使用更小的初始化范围"""
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight, gain=0.1)  # 使用较小的增益
                nn.init.zeros_(layer.bias)
    
    def forward(self, x):
        """
        前向传播
        输入: x [batch, 3] - 无量纲坐标 (x*, y*, τ*)
        输出: u, v, p, T
        """
        # 网络预测
        out = self.net(x)
        
        # 分割输出并应用激活函数以确保物理合理性
        u = torch.tanh(out[:, 0:1]) * self.U_char  # 限制速度大小
        v = torch.tanh(out[:, 1:2]) * self.U_char
        p = out[:, 2:3] * 0.1 * self.p_char  # 减小压力幅度
        
        # 温度在合理范围内 (290-360K)
        T = self.T0 + (self.Tw_heat - self.T0) * torch.sigmoid(out[:, 3:4])
        
        return u, v, p, T

# ==================== 简化的拓扑网络 ====================
class SimpleTopologyNetwork(nn.Module):
    """简化的拓扑设计网络 - 更稳定"""
    
    def __init__(self, hidden_dim=128, num_layers=4):
        super(SimpleTopologyNetwork, self).__init__()
        
        # 构建网络
        layers = []
        layers.append(nn.Linear(2, hidden_dim))
        layers.append(nn.Tanh())
        
        for _ in range(num_layers - 2):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        
        layers.append(nn.Linear(hidden_dim, 1))
        layers.append(nn.Sigmoid())  # 输出在[0,1]之间
        
        self.net = nn.Sequential(*layers)
        
        # 初始化权重
        self._initialize_weights()
    
    def _initialize_weights(self):
        """初始化权重"""
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight, gain=1.0)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)
    
    def forward(self, x_space):
        """
        前向传播
        输入: x_space [batch, 2] - 无量纲空间坐标
        输出: rho [batch, 1] - 拓扑设计变量 ∈ [0,1]
        """
        return self.net(x_space)

# ==================== 材料模型 - 更稳定 ====================
class StableMaterialModel:
    """稳定的材料模型"""
    
    def __init__(self, device='cpu'):
        self.device = device
        
        # PCM材料参数
        self.Tpc = Tpc
        self.DeltaT = DeltaT
        self.L = L
        self.rho_s = rho_s
        self.rho_l = rho_l
        self.cp_s = cp_s
        self.cp_l = cp_l
        self.lambda_s = lambda_s
        self.lambda_l = lambda_l
        self.mu_l = mu_l
        
        # 铜的材料参数
        self.rho_Cu = rho_Cu
        self.lambda_Cu = lambda_Cu
        self.cp_Cu = cp_Cu
        
        # 特征尺度
        self.L_char = L_char
        self.U_char = U_char
        self.t_char = t_char
        self.T_char = T_char
        self.p_char = p_char
        
    def compute_liquid_fraction(self, T):
        """计算液相率 - 使用稳定的过渡函数"""
        if isinstance(T, torch.Tensor):
            # 使用平滑的过渡函数
            normalized = (T - self.Tpc) / (self.DeltaT / 2.0)
            # 使用更稳定的sigmoid函数
            phi = torch.sigmoid(normalized)
            return phi
        else:
            normalized = (T - self.Tpc) / (self.DeltaT / 2.0)
            phi = 1.0 / (1.0 + np.exp(-normalized))
            return np.clip(phi, 0.0, 1.0)
    
    def compute_effective_cp(self, T):
        """计算有效比热容（包含潜热）"""
        phi = self.compute_liquid_fraction(T)
        
        if isinstance(T, torch.Tensor):
            # 显热部分
            cp_sensible = self.cp_s + (self.cp_l - self.cp_s) * phi
            
            # 潜热部分 - 使用更稳定的高斯函数
            sigma = self.DeltaT / 4.0  # 标准差
            exponent = -((T - self.Tpc) ** 2) / (2 * sigma ** 2 + eps)
            gaussian = torch.exp(exponent) / (sigma * np.sqrt(2 * np.pi) + eps)
            
            cp_latent = self.L * gaussian
            
            return cp_sensible + cp_latent, phi
        else:
            cp_sensible = self.cp_s + (self.cp_l - self.cp_s) * phi
            
            sigma = self.DeltaT / 4.0
            exponent = -((T - self.Tpc) ** 2) / (2 * sigma ** 2 + eps)
            gaussian = np.exp(exponent) / (sigma * np.sqrt(2 * np.pi) + eps)
            
            cp_latent = self.L * gaussian
            
            return cp_sensible + cp_latent, phi
    
    def compute_properties(self, T, rho_design):
        """计算混合材料属性 - 简化稳定的混合规则"""
        # 计算PCM属性
        cp_eff, phi = self.compute_effective_cp(T)
        
        if isinstance(T, torch.Tensor):
            # PCM密度 - 简化计算
            rho_pcm = self.rho_s * (1 - phi) + self.rho_l * phi
            
            # PCM导热系数 - 简化计算
            lambda_pcm = self.lambda_s * (1 - phi) + self.lambda_l * phi
            
            # PCM粘度 - 简化计算
            mu_pcm = self.mu_l * phi + 1e6 * (1 - phi)  # 固相粘度更大
            
            # 确保rho_design在[0,1]范围内
            rho_design = torch.clamp(rho_design, 0.0, 1.0)
            
            # 简单线性混合（更稳定）
            rho_total = rho_design * self.rho_Cu + (1 - rho_design) * rho_pcm
            lambda_total = rho_design * self.lambda_Cu + (1 - rho_design) * lambda_pcm
            
            # 质量加权平均比热容
            mass_Cu = rho_design * self.rho_Cu
            mass_pcm = (1 - rho_design) * rho_pcm
            mass_total = mass_Cu + mass_pcm + eps
            cp_total = (mass_Cu * self.cp_Cu + mass_pcm * cp_eff) / mass_total
            
            # 混合粘度
            mu_total = rho_design * 1e6 + (1 - rho_design) * mu_pcm
            
        else:
            # NumPy版本
            rho_pcm = self.rho_s * (1 - phi) + self.rho_l * phi
            lambda_pcm = self.lambda_s * (1 - phi) + self.lambda_l * phi
            mu_pcm = self.mu_l * phi + 1e6 * (1 - phi)
            
            rho_design = np.clip(rho_design, 0.0, 1.0)
            
            rho_total = rho_design * self.rho_Cu + (1 - rho_design) * rho_pcm
            lambda_total = rho_design * self.lambda_Cu + (1 - rho_design) * lambda_pcm
            
            mass_Cu = rho_design * self.rho_Cu
            mass_pcm = (1 - rho_design) * rho_pcm
            mass_total = mass_Cu + mass_pcm + eps
            cp_total = (mass_Cu * self.cp_Cu + mass_pcm * cp_eff) / mass_total
            
            mu_total = rho_design * 1e6 + (1 - rho_design) * mu_pcm
        
        # 计算其他属性
        nu_total = mu_total / (rho_total + eps)
        a_total = lambda_total / (rho_total * cp_total + eps)
        
        return rho_total, lambda_total, cp_total, mu_total, nu_total, a_total, phi, rho_design

# ==================== 稳定的TopoPINN模型 ====================
class StableTopoPINN(nn.Module):
    """稳定的拓扑优化PINN模型"""
    
    def __init__(self, hidden_dim=128, num_layers=6):
        super(StableTopoPINN, self).__init__()
        
        # 物理场网络
        self.physics_nn = StablePhysicsInformedNN(
            input_dim=3,
            hidden_dim=hidden_dim, 
            num_layers=num_layers
        )
        
        # 拓扑网络
        self.topology_nn = SimpleTopologyNetwork(
            hidden_dim=hidden_dim,
            num_layers=num_layers
        )
        
        # 材料模型
        self.material = StableMaterialModel()
        
        # 物理参数
        self.register_buffer('g_val', torch.tensor(g))
        self.register_buffer('beta', torch.tensor(alpha))
        self.register_buffer('Tpc', torch.tensor(Tpc))
        
        # 过程标志
        self.is_heating = True  # 默认储热过程
    
    def set_heating_process(self, is_heating=True):
        """设置过程类型：储热或放热"""
        self.is_heating = is_heating
    
    def forward(self, x):
        """
        前向传播
        输入: x [batch, 3] - 无量纲坐标 (x*, y*, τ*)
        输出: u, v, p, T, rho
        """
        # 提取空间坐标
        x_space = x[:, 0:2]
        
        # 物理场预测
        u, v, p, T = self.physics_nn(x)
        
        # 拓扑设计变量
        rho = self.topology_nn(x_space)
        
        return u, v, p, T, rho
    
    def compute_pde_residuals(self, x):
        """
        计算PDE残差 - 简化的稳定版本
        """
        # 确保输入需要梯度
        if not x.requires_grad:
            x = x.clone().requires_grad_(True)
        
        # 前向传播
        u, v, p, T, rho = self(x)
        
        # 计算材料属性
        rho_total, lambda_total, cp_total, mu_total, nu_total, a_total, phi, rho_design = self.material.compute_properties(T, rho)
        
        # ========== 计算梯度 ==========
        # 分别计算每个输出的梯度
        gradients = []
        
        # 计算速度梯度
        if u.requires_grad:
            grad_u = torch.autograd.grad(
                u, x, 
                grad_outputs=torch.ones_like(u),
                create_graph=True, 
                retain_graph=True,
                allow_unused=True
            )[0]
            if grad_u is None:
                grad_u = torch.zeros_like(x)
        else:
            grad_u = torch.zeros_like(x)
        
        # 计算温度梯度
        if T.requires_grad:
            grad_T = torch.autograd.grad(
                T, x, 
                grad_outputs=torch.ones_like(T),
                create_graph=True, 
                retain_graph=True,
                allow_unused=True
            )[0]
            if grad_T is None:
                grad_T = torch.zeros_like(x)
        else:
            grad_T = torch.zeros_like(x)
        
        # 转换为实际导数
        dudx = grad_u[:, 0:1] / self.physics_nn.L_char
        dudy = grad_u[:, 1:2] / self.physics_nn.L_char
        dudt = grad_u[:, 2:3] / self.physics_nn.t_char
        
        # v的梯度（假设与u类似，简化处理）
        dvdx = torch.zeros_like(dudx)
        dvdy = torch.zeros_like(dudy)
        dvdt = torch.zeros_like(dudt)
        
        dTdx = grad_T[:, 0:1] / self.physics_nn.L_char
        dTdy = grad_T[:, 1:2] / self.physics_nn.L_char
        dTdt = grad_T[:, 2:3] / self.physics_nn.t_char
        
        # 计算二阶导数 - 简化的数值计算
        # 使用一阶差分的差分来近似二阶导数
        d2udx2 = torch.zeros_like(dudx)
        d2udy2 = torch.zeros_like(dudy)
        d2Tdx2 = torch.zeros_like(dTdx)
        d2Tdy2 = torch.zeros_like(dTdy)
        
        # ========== 质量守恒方程 ==========
        # ∇·u = 0 (不可压缩)
        mass_residual = dudx + dvdy
        
        # ========== 动量守恒方程 ==========
        # x方向: ∂u/∂t + u·∇u = -1/ρ ∂p/∂x + ν∇²u
        # 简化处理：主要考虑浮力驱动
        buoyancy_x = torch.zeros_like(u)
        buoyancy_y = self.beta * self.g_val * (T - self.Tpc)
        
        # 简化动量方程残差
        mom_x_residual = dudt - nu_total * (d2udx2 + d2udy2) + buoyancy_x
        mom_y_residual = dvdt - nu_total * (d2udx2 + d2udy2) + buoyancy_y
        
        # ========== 能量守恒方程 ==========
        # ∂T/∂t + u·∇T = a∇²T
        # 简化对流项
        conv_T = u * dTdx + v * dTdy
        diffusion_T = a_total * (d2Tdx2 + d2Tdy2)
        
        energy_residual = dTdt + conv_T - diffusion_T
        
        # 简化压力梯度（通过浮力体现）
        
        return {
            'mass': mass_residual,
            'mom_x': mom_x_residual,
            'mom_y': mom_y_residual,
            'energy': energy_residual,
            'u': u, 'v': v, 'p': p, 'T': T,
            'rho_design': rho_design,
            'phi': phi,
            'rho_total': rho_total,
            'lambda_total': lambda_total,
            'nu_total': nu_total
        }

# ==================== 采样器 ====================
class StableSampler:
    """稳定的采样器"""
    
    def __init__(self, r0, r1, t_char):
        self.r0 = r0
        self.r1 = r1
        self.t_char = t_char
        
    def sample_domain(self, N, time_dependent=True):
        """采样计算域内部点"""
        # 极坐标采样
        r = np.sqrt(np.random.uniform(r0**2, r1**2, N))
        theta = np.random.uniform(0, 2*np.pi, N)
        
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        
        # 时间采样
        if time_dependent:
            t = np.random.uniform(0, self.t_char, N)
        else:
            t = np.full(N, 0.5 * self.t_char)
        
        # 无量纲化
        x_star = x / L_char
        y_star = y / L_char
        tau_star = t / self.t_char
        
        points = np.column_stack([x_star, y_star, tau_star])
        
        return torch.tensor(points, dtype=torch.float32)
    
    def sample_boundary(self, N, boundary='inner', time_dependent=True):
        """采样边界点"""
        if boundary == 'inner':
            r = self.r0
        else:
            r = self.r1
        
        theta = np.random.uniform(0, 2*np.pi, N)
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        
        if time_dependent:
            t = np.random.uniform(0, self.t_char, N)
        else:
            t = np.zeros(N)
        
        # 无量纲化
        x_star = x / L_char
        y_star = y / L_char
        tau_star = t / self.t_char
        
        points = np.column_stack([x_star, y_star, tau_star])
        
        return torch.tensor(points, dtype=torch.float32)
    
    def sample_initial(self, N):
        """采样初始条件点"""
        r = np.sqrt(np.random.uniform(r0**2, r1**2, N))
        theta = np.random.uniform(0, 2*np.pi, N)
        
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        t = np.zeros(N)
        
        # 无量纲化
        x_star = x / L_char
        y_star = y / L_char
        tau_star = t / self.t_char
        
        points = np.column_stack([x_star, y_star, tau_star])
        
        return torch.tensor(points, dtype=torch.float32)

# ==================== 损失计算器 ====================
class StableLossCalculator:
    """稳定的损失计算器"""
    
    def __init__(self, model):
        self.model = model
        self.sampler = StableSampler(r0, r1, t_char)
        
        # 稳定的权重设置
        self.weights = {
            'pde_mass': 0.1,
            'pde_mom_x': 0.1,
            'pde_mom_y': 0.1,
            'pde_energy': 1.0,
            'ic': 10.0,
            'bc_inner': 50.0,
            'bc_outer': 10.0,
            'volume': 5.0,
            'objective': 2.0
        }
        
        # 当前体积分数
        self.current_volume = 0.0
        
        # 损失历史
        self.history = {key: [] for key in self.weights.keys()}
        self.history['total_loss'] = []
    
    def compute_total_loss(self, epoch=0, stage='physics_only'):
        """计算总损失"""
        # 采样点
        N_domain = 500
        N_boundary = 100
        N_initial = 100
        
        x_domain = self.sampler.sample_domain(N_domain).to(device)
        x_inner = self.sampler.sample_boundary(N_boundary, 'inner').to(device)
        x_outer = self.sampler.sample_boundary(N_boundary, 'outer').to(device)
        x_initial = self.sampler.sample_initial(N_initial).to(device)
        
        # 计算各项损失
        losses = {}
        
        # PDE损失 - 分开各项
        residuals = self.model.compute_pde_residuals(x_domain)
        losses['pde_mass'] = torch.mean(torch.abs(residuals['mass']))
        losses['pde_mom_x'] = torch.mean(torch.abs(residuals['mom_x']))
        losses['pde_mom_y'] = torch.mean(torch.abs(residuals['mom_y']))
        losses['pde_energy'] = torch.mean(torch.abs(residuals['energy']))
        
        # 初始条件损失
        losses['ic'] = self.compute_ic_loss(x_initial)
        
        # 边界条件损失
        losses['bc_inner'] = self.compute_bc_loss(x_inner, boundary='inner')
        losses['bc_outer'] = self.compute_bc_loss(x_outer, boundary='outer')
        
        # 体积约束损失
        if stage == 'physics_only':
            losses['volume'] = torch.tensor(0.0).to(device)
        else:
            losses['volume'] = self.compute_volume_loss(x_domain)
        
        # 优化目标损失
        if stage == 'physics_only':
            losses['objective'] = torch.tensor(0.0).to(device)
        else:
            losses['objective'] = self.compute_objective_loss(x_domain, residuals)
        
        # 记录历史
        for key in losses:
            self.history[key].append(losses[key].item())
        
        # 计算加权总损失
        total_loss = torch.tensor(0.0).to(device)
        for key, loss in losses.items():
            total_loss += self.weights[key] * loss
        
        self.history['total_loss'].append(total_loss.item())
        
        return total_loss, losses
    
    def compute_ic_loss(self, x):
        """计算初始条件损失"""
        u, v, p, T, rho = self.model(x)
        
        # 初始温度应为T0，速度应为0
        T0 = self.model.physics_nn.T0
        T_loss = torch.mean(torch.abs(T - T0))
        
        # 简化速度损失
        u_loss = torch.mean(torch.abs(u))
        v_loss = torch.mean(torch.abs(v))
        
        return T_loss + 0.01 * (u_loss + v_loss)
    
    def compute_bc_loss(self, x, boundary='inner'):
        """计算边界条件损失"""
        u, v, p, T, rho = self.model(x)
        
        if boundary == 'inner':
            # 内壁：恒温边界
            if self.model.is_heating:
                Tw = self.model.physics_nn.Tw_heat  # 储热时内壁温度
            else:
                Tw = self.model.physics_nn.Tw_cool  # 放热时内壁温度
            
            T_loss = torch.mean(torch.abs(T - Tw))
            
            # 无滑移边界条件
            u_loss = torch.mean(torch.abs(u))
            v_loss = torch.mean(torch.abs(v))
            
            return T_loss + 0.01 * (u_loss + v_loss)
        else:
            # 外壁：绝热边界 - 简化处理
            # 使用温度梯度近似
            return torch.tensor(0.0).to(device)
    
    def compute_volume_loss(self, x):
        """计算体积约束损失"""
        x_space = x[:, 0:2]
        rho = self.model.topology_nn(x_space)
        
        # 计算平均体积分数
        volume = torch.mean(rho)
        self.current_volume = volume.item()
        
        # 体积约束损失 - 使用软约束
        violation = torch.abs(volume - phi_total)
        return violation
    
    def compute_objective_loss(self, x, residuals=None):
        """计算优化目标损失"""
        if residuals is None:
            residuals = self.model.compute_pde_residuals(x)
        
        T = residuals['T']
        
        if case == 1:
            # 最小化平均温度
            T_avg = torch.mean(T)
            # 目标是使温度接近相变温度，以充分利用潜热
            target_temp = Tpc + 2.0  # 略高于相变温度
            return torch.abs(T_avg - target_temp)
        elif case == 2:
            # 最小化温度方差 - 促进均匀性
            T_avg = torch.mean(T)
            T_var = torch.mean(torch.abs(T - T_avg))
            return T_var
        else:
            # 多目标优化
            T_avg = torch.mean(T)
            T_var = torch.mean(torch.abs(T - T_avg))
            
            # 简化处理：组合目标
            return 0.5 * torch.abs(T_avg - 320.0) + 0.5 * T_var

# ==================== 训练器 ====================
class StableTrainer:
    """稳定的训练器"""
    
    def __init__(self, model):
        self.model = model
        self.loss_calculator = StableLossCalculator(model)
        
        # 简化的训练阶段
        self.stages = [
            {'name': 'physics_only', 'epochs': 50, 'lr': 1e-4, 'desc': '仅训练物理场'},
            {'name': 'joint_training', 'epochs': 100, 'lr': 5e-5, 'desc': '联合训练'},
        ]
        
        # 训练历史
        self.history = {
            'total_loss': [],
            'volume': [],
            'stage': [],
            'learning_rate': []
        }
        
        # 创建保存目录
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.save_dir = f"./results/case{case}_{timestamp}"
        os.makedirs(self.save_dir, exist_ok=True)
    
    def train(self):
        """训练模型"""
        print("="*60)
        print("开始训练拓扑优化PINN - 稳定版")
        print("="*60)
        
        total_epochs = 0
        
        for stage_idx, stage_config in enumerate(self.stages):
            stage_name = stage_config['name']
            epochs = stage_config['epochs']
            lr = stage_config['lr']
            
            print(f"\n{'='*60}")
            print(f"阶段 {stage_idx+1}/{len(self.stages)}: {stage_config['desc']}")
            print(f"轮次: {epochs}, 学习率: {lr}")
            print(f"{'='*60}")
            
            # 设置优化器
            if stage_name == 'physics_only':
                # 只训练物理场网络
                params = list(self.model.physics_nn.parameters())
                for param in self.model.topology_nn.parameters():
                    param.requires_grad = False
            else:
                # 联合训练所有参数
                params = self.model.parameters()
                for param in self.model.parameters():
                    param.requires_grad = True
            
            optimizer = optim.Adam(params, lr=lr)
            # 移除verbose参数以兼容旧版PyTorch
            scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
            
            # 阶段训练
            for epoch in range(epochs):
                total_epochs += 1
                
                try:
                    # 计算损失
                    total_loss, losses = self.loss_calculator.compute_total_loss(
                        epoch=total_epochs, stage=stage_name
                    )
                    
                    # 检查NaN
                    if torch.isnan(total_loss) or torch.isinf(total_loss):
                        print(f"警告: 第{epoch+1}轮损失为NaN或inf，跳过本轮")
                        continue
                    
                    # 反向传播
                    optimizer.zero_grad()
                    total_loss.backward()
                    
                    # 梯度裁剪
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    
                    # 优化
                    optimizer.step()
                    scheduler.step(total_loss)
                    
                    # 记录历史
                    self.history['total_loss'].append(total_loss.item())
                    self.history['volume'].append(self.loss_calculator.current_volume)
                    self.history['stage'].append(stage_idx)
                    self.history['learning_rate'].append(optimizer.param_groups[0]['lr'])
                    
                    # 输出进度
                    if (epoch + 1) % 10 == 0:
                        print(f"Epoch {total_epochs:4d} | Loss: {total_loss.item():.4e} | "
                              f"Volume: {self.loss_calculator.current_volume:.4f} | "
                              f"PDE_E: {losses['pde_energy'].item():.2e} | "
                              f"BC: {losses['bc_inner'].item():.2e} | "
                              f"LR: {optimizer.param_groups[0]['lr']:.2e}")
                    
                    # 保存检查点
                    if (epoch + 1) % 20 == 0:
                        self.save_checkpoint(f"{stage_name}_epoch{epoch+1}")
                
                except Exception as e:
                    print(f"训练出错: {e}")
                    import traceback
                    traceback.print_exc()
                    break
            
            # 保存阶段检查点
            self.save_checkpoint(stage_name)
        
        print("\n训练完成!")
        self.save_results()
    
    def save_checkpoint(self, checkpoint_name):
        """保存检查点"""
        checkpoint = {
            'model_state_dict': self.model.state_dict(),
            'history': self.history,
            'char_scales': char_scales,
            'loss_history': self.loss_calculator.history
        }
        
        filename = f"{self.save_dir}/checkpoint_{checkpoint_name}.pth"
        torch.save(checkpoint, filename)
    
    def save_results(self):
        """保存结果"""
        # 保存最终模型
        torch.save(self.model.state_dict(), f"{self.save_dir}/model_final.pth")
        
        # 保存训练历史
        np.savez(f"{self.save_dir}/training_history.npz",
                 total_loss=self.history['total_loss'],
                 volume=self.history['volume'],
                 stage=self.history['stage'],
                 learning_rate=self.history['learning_rate'])
        
        # 保存损失历史
        for key in self.loss_calculator.history:
            np.save(f"{self.save_dir}/loss_{key}.npy", np.array(self.loss_calculator.history[key]))
        
        # 绘制训练曲线
        self.plot_training_history()
        
        print(f"\n所有结果已保存到: {self.save_dir}")
    
    def plot_training_history(self):
        """绘制训练历史曲线"""
        if not self.history['total_loss']:
            print("警告: 没有训练历史数据")
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        
        # 总损失
        axes[0, 0].semilogy(self.history['total_loss'])
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Total Loss')
        axes[0, 0].set_title('Total Loss History')
        axes[0, 0].grid(True, alpha=0.3)
        
        # 体积分数
        axes[0, 1].plot(self.history['volume'])
        axes[0, 1].axhline(y=phi_total, color='r', linestyle='--', label=f'Target: {phi_total}')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Volume Fraction')
        axes[0, 1].set_title('Volume Constraint')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # PDE损失分量
        pde_keys = ['pde_mass', 'pde_mom_x', 'pde_mom_y', 'pde_energy']
        colors = plt.cm.tab10(np.linspace(0, 1, len(pde_keys)))
        for i, key in enumerate(pde_keys):
            if key in self.loss_calculator.history and self.loss_calculator.history[key]:
                axes[1, 0].semilogy(self.loss_calculator.history[key], 
                                  label=key, color=colors[i], alpha=0.7)
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Loss')
        axes[1, 0].set_title('PDE Loss Components')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # 边界和初始条件损失
        bc_keys = ['bc_inner', 'bc_outer', 'ic']
        for i, key in enumerate(bc_keys):
            if key in self.loss_calculator.history and self.loss_calculator.history[key]:
                axes[1, 1].semilogy(self.loss_calculator.history[key], 
                                  label=key, alpha=0.7)
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Loss')
        axes[1, 1].set_title('Boundary & Initial Conditions')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f"{self.save_dir}/training_history.png", dpi=300)
        plt.close()

# ==================== 验证函数 ====================
def validate_model(model):
    """验证模型"""
    print("\n" + "="*60)
    print("模型验证")
    print("="*60)
    
    sampler = StableSampler(r0, r1, t_char)
    
    # 采样验证点
    x_val = sampler.sample_domain(200).to(device)
    x_inner = sampler.sample_boundary(50, 'inner').to(device)
    x_initial = sampler.sample_initial(50).to(device)
    
    with torch.no_grad():
        # 验证边界条件
        u_inner, v_inner, p_inner, T_inner, rho_inner = model(x_inner)
        
        if model.is_heating:
            Tw = model.physics_nn.Tw_heat
        else:
            Tw = model.physics_nn.Tw_cool
            
        T_error_inner = torch.mean(torch.abs(T_inner - Tw)).item()
        u_error_inner = torch.mean(torch.abs(u_inner)).item()
        print(f"内壁温度误差: {T_error_inner:.4f} K")
        print(f"内壁速度误差: {u_error_inner:.4f} m/s")
        
        # 验证初始条件
        u_initial, v_initial, p_initial, T_initial, rho_initial = model(x_initial)
        T_error_initial = torch.mean(torch.abs(T_initial - 290.0)).item()
        print(f"初始温度误差: {T_error_initial:.4f} K")
        
        # 验证体积约束
        x_space = x_val[:, 0:2]
        rho = model.topology_nn(x_space)
        volume = torch.mean(rho).item()
        volume_min = rho.min().item()
        volume_max = rho.max().item()
        print(f"\n体积分数统计:")
        print(f"  平均值: {volume:.4f} (目标: {phi_total})")
        print(f"  范围: [{volume_min:.3f}, {volume_max:.3f}]")
        
        # 验证物理合理性
        print(f"\n物理量范围:")
        print(f"  温度范围: {T_initial.min().item():.1f}K - {T_initial.max().item():.1f}K")
        print(f"  速度范围: u[{u_initial.min().item():.4f}, {u_initial.max().item():.4f}] m/s")
    
    # 计算PDE残差
    print("\n计算PDE残差...")
    x_val.requires_grad_(True)
    residuals = model.compute_pde_residuals(x_val)
    
    print("PDE残差:")
    print(f"  质量守恒: {torch.mean(torch.abs(residuals['mass'])).item():.4e}")
    print(f"  x动量守恒: {torch.mean(torch.abs(residuals['mom_x'])).item():.4e}")
    print(f"  y动量守恒: {torch.mean(torch.abs(residuals['mom_y'])).item():.4e}")
    print(f"  能量守恒: {torch.mean(torch.abs(residuals['energy'])).item():.4e}")
    
    print("="*60)
    
    return residuals

# ==================== 可视化函数 ====================
def visualize_results(model, save_dir, num_points=1000):
    """可视化结果"""
    os.makedirs(save_dir, exist_ok=True)
    
    sampler = StableSampler(r0, r1, t_char)
    
    # 不同时间点的可视化
    time_points = [0.0, 0.25, 0.5, 0.75, 1.0]
    
    for t_frac in time_points:
        # 采样点
        x_sample = sampler.sample_domain(num_points, time_dependent=False).to(device)
        # 设置时间
        x_sample[:, 2] = t_frac
        
        with torch.no_grad():
            u, v, p, T, rho = model(x_sample)
        
        # 提取坐标和结果
        x_real = x_sample[:, 0].cpu().numpy() * L_char
        y_real = x_sample[:, 1].cpu().numpy() * L_char
        T_vals = T.cpu().numpy().flatten()
        rho_vals = rho.cpu().numpy().flatten()
        u_mag = torch.sqrt(u**2 + v**2).cpu().numpy().flatten()
        
        # 绘图
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        
        # 温度场
        sc1 = axes[0, 0].scatter(x_real, y_real, c=T_vals, cmap='jet', s=10, vmin=290, vmax=360)
        axes[0, 0].set_title(f'Temperature Distribution (t={t_frac:.2f})')
        axes[0, 0].set_xlabel('x (m)')
        axes[0, 0].set_ylabel('y (m)')
        axes[0, 0].axis('equal')
        plt.colorbar(sc1, ax=axes[0, 0])
        
        # 拓扑结构
        sc2 = axes[0, 1].scatter(x_real, y_real, c=rho_vals, cmap='binary', s=10, vmin=0, vmax=1)
        axes[0, 1].set_title(f'Topology Distribution (t={t_frac:.2f})')
        axes[0, 1].set_xlabel('x (m)')
        axes[0, 1].set_ylabel('y (m)')
        axes[0, 1].axis('equal')
        plt.colorbar(sc2, ax=axes[0, 1])
        
        # 速度场
        sc3 = axes[1, 0].scatter(x_real, y_real, c=u_mag, cmap='plasma', s=10)
        axes[1, 0].set_title(f'Velocity Magnitude (t={t_frac:.2f})')
        axes[1, 0].set_xlabel('x (m)')
        axes[1, 0].set_ylabel('y (m)')
        axes[1, 0].axis('equal')
        plt.colorbar(sc3, ax=axes[1, 0])
        
        # 拓扑结构直方图
        axes[1, 1].hist(rho_vals, bins=30, color='skyblue', edgecolor='black', alpha=0.7)
        axes[1, 1].axvline(x=phi_total, color='red', linestyle='--', linewidth=2, label=f'Target: {phi_total}')
        axes[1, 1].set_xlabel('Topology Value')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].set_title(f'Topology Value Distribution (t={t_frac:.2f})')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f"{save_dir}/results_t{t_frac:.2f}.png", dpi=300)
        plt.close()
    
    print(f"可视化结果已保存到: {save_dir}")

# ==================== 主程序 ====================
if __name__ == "__main__":
    print("拓扑优化PINN - 相变储热系统（稳定优化版）")
    print("="*60)
    
    # 1. 初始化稳定模型
    print("初始化稳定模型...")
    model = StableTopoPINN(
        hidden_dim=128,
        num_layers=6
    ).to(device)
    
    # 设置储热过程
    model.set_heating_process(True)
    
    # 2. 验证初始模型
    print("验证初始模型...")
    validate_model(model)
    
    # 3. 训练模型
    print("\n开始训练...")
    trainer = StableTrainer(model)
    trainer.train()
    
    # 4. 验证训练后模型
    print("\n验证训练后模型...")
    residuals = validate_model(model)
    
    # 5. 可视化结果
    print("\n生成可视化结果...")
    visualize_results(model, save_dir=f"{trainer.save_dir}/visualization", num_points=2000)
    
    # 6. 分析结果
    print("\n分析优化结果...")
    with torch.no_grad():
        # 采样分析点
        sampler = StableSampler(r0, r1, t_char)
        x_analysis = sampler.sample_domain(1000).to(device)
        u, v, p, T, rho = model(x_analysis)
        
        # 计算性能指标
        T_avg = torch.mean(T).item()
        T_std = torch.std(T).item()
        volume_actual = torch.mean(rho).item()
        
        print(f"\n性能指标:")
        print(f"  平均温度: {T_avg:.2f} K")
        print(f"  温度标准差: {T_std:.2f} K")
        print(f"  实际体积分数: {volume_actual:.4f}")
        print(f"  目标体积分数: {phi_total}")
        print(f"  体积误差: {abs(volume_actual - phi_total):.4f}")
        
        # 相变利用情况
        in_phase_change = torch.sum((T > Tpc - DeltaT/2) & (T < Tpc + DeltaT/2)).item()
        phase_change_ratio = in_phase_change / len(T)
        print(f"  相变区间占比: {phase_change_ratio:.3f}")
    
    print("\n" + "="*60)
    print("程序执行完成!")
    print(f"结果保存目录: {trainer.save_dir}")
    print("="*60)

使用设备: cuda
修正的特征尺度:
特征长度 L_char = 0.0800 m
特征速度 U_char = 0.062642 m/s
特征时间 t_char = 30694.40 s
特征温差 T_char = 50.0 K
特征压力 p_char = 3.453120 Pa
拓扑优化PINN - 相变储热系统（稳定优化版）
初始化稳定模型...
验证初始模型...

模型验证
内壁温度误差: 35.0000 K
内壁速度误差: 0.0000 m/s
初始温度误差: 35.0000 K

体积分数统计:
  平均值: 0.5010 (目标: 0.3)
  范围: [0.473, 0.528]

物理量范围:
  温度范围: 325.0K - 325.0K
  速度范围: u[-0.0000, 0.0000] m/s

计算PDE残差...
PDE残差:
  质量守恒: 2.2656e-07
  x动量守恒: 3.4047e-14
  y动量守恒: 8.6819e-03
  能量守恒: 1.6734e-11

开始训练...
开始训练拓扑优化PINN - 稳定版

阶段 1/2: 仅训练物理场
轮次: 50, 学习率: 0.0001
Epoch   10 | Loss: 2.0982e+03 | Volume: 0.0000 | PDE_E: 7.67e-09 | BC: 3.50e+01 | LR: 1.00e-04
Epoch   20 | Loss: 2.0955e+03 | Volume: 0.0000 | PDE_E: 7.47e-08 | BC: 3.49e+01 | LR: 1.00e-04
Epoch   30 | Loss: 2.0908e+03 | Volume: 0.0000 | PDE_E: 4.37e-07 | BC: 3.48e+01 | LR: 1.00e-04
Epoch   40 | Loss: 2.0818e+03 | Volume: 0.0000 | PDE_E: 1.89e-06 | BC: 3.46e+01 | LR: 1.00e-04
Epoch   50 | Loss: 2.0615e+03 | Volume: 0.0000 | PDE_E: 6.31e-06 | BC: 3.41e+01 | LR: 1.00e-0